# Lab 3b: VDB Corpus Acquisition with FAISS
**Course:** CIS 531/731 | **Module 3: Obtain**

## Objective
Take the text output generated by the Whisper ASR pipeline, convert it into semantic embeddings using a HuggingFace Transformer, and index it in a local Vector Database (FAISS).

### DevContainer Setup Instructions
Your `verify_env.py` script from MP2 already confirmed the presence of `faiss` and `transformers`. If your DevContainer failed to build those, rebuild the container now. If you are running locally outside the container, you must run:
`!pip install faiss-cpu transformers sentence-transformers`

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"FAISS Version: {faiss.__version__}")

### 1. Initialize the Embedding Model
We use `all-MiniLM-L6-v2` as a lightweight, highly efficient sentence embedding model to encode our transcriptions into high-dimensional vectors.

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vector_dimension = embedder.get_sentence_embedding_dimension()
print(f"Model loaded. Embedding vector dimension: {vector_dimension}")

### 2. Create the FAISS Index & Ingest Corpus
Here we simulate the ingestion of chunks coming from our live radio stream pipeline.

In [ ]:
# Initialize a flat L2 (Euclidean distance) FAISS index
index = faiss.IndexFlatL2(vector_dimension)

# Simulated incoming ASR corpus chunks
corpus = [
    "The caller sounds really upset about their recent breakup.",
    "I love this Cage the Elephant track, it reminds me of summer.",
    "Sometimes I feel like our whole generation is just out of time."
]

# Encode the text into vectors
corpus_embeddings = embedder.encode(corpus)

# Add vectors to the VDB
index.add(np.array(corpus_embeddings).astype('float32'))
print(f"Successfully ingested {index.ntotal} vectors into FAISS.")

### 3. Query the VDB (Mapping Subtle Tropes)
We query the database for a semantic concept ("doomed youth") to see which transcribed chunk matches closest in latent space.

In [ ]:
query = "doomed youth"
query_embedding = embedder.encode([query]).astype('float32')

# Search the top 1 most similar result (k=1)
distances, indices = index.search(query_embedding, k=1)

matched_index = indices[0][0]
print(f"Query: '{query}'")
print(f"Closest Match in VDB: '{corpus[matched_index]}'")
print(f"L2 Distance: {distances[0][0]:.4f}")